# Jackfruit Surface Analysis
วิเคราะห์หนามและตาขนุนด้วย OpenCV (ไม่ต้อง train model ใหม่)

## Step 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive'
CLASSES = ['ขนุนดิบ', 'ขนุนสุก']

## Step 2 — Import

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd
print('Import สำเร็จ')

## Step 3 — ฟังก์ชันวิเคราะห์ผิวขนุน

In [ ]:
def analyze_jackfruit_surface(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None

    img = cv2.resize(img, (224, 224))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 1. ตรวจจับสีดำของตาขนุน
    black_mask = cv2.inRange(hsv,
        np.array([0, 0, 0]),
        np.array([180, 255, 60]))
    black_ratio = np.sum(black_mask > 0) / black_mask.size

    # 2. ตรวจจับสีเขียวของเปลือก
    green_mask = cv2.inRange(hsv,
        np.array([25, 30, 30]),
        np.array([85, 255, 255]))
    green_ratio = np.sum(green_mask > 0) / green_mask.size

    # 3. ตรวจจับสีเหลือง
    yellow_mask = cv2.inRange(hsv,
        np.array([15, 30, 30]),
        np.array([35, 255, 255]))
    yellow_ratio = np.sum(yellow_mask > 0) / yellow_mask.size

    # 4. นับหนามด้วย contour
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 30, 100)
    contours, _ = cv2.findContours(
        edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # กรองเฉพาะ contour ขนาดพอเหมาะ (หนาม)
    spine_contours = [c for c in contours
                      if 10 < cv2.contourArea(c) < 500]
    spine_count = len(spine_contours)

    # 5. ความหยาบของผิว (texture)
    texture = gray.std()

    return {
        'black_ratio': round(black_ratio, 4),
        'green_ratio': round(green_ratio, 4),
        'yellow_ratio': round(yellow_ratio, 4),
        'spine_count': spine_count,
        'texture': round(texture, 2)
    }

print('ฟังก์ชันพร้อมแล้ว')

## Step 4 — วิเคราะห์รูปทั้งหมด

In [ ]:
results = []

for label, class_name in enumerate(CLASSES):
    class_path = os.path.join(DATASET_PATH, class_name)
    files = os.listdir(class_path)
    print(f'กำลังวิเคราะห์ {class_name}: {len(files)} รูป...')

    for fname in files:
        fpath = os.path.join(class_path, fname)
        features = analyze_jackfruit_surface(fpath)
        if features:
            features['class'] = class_name
            features['label'] = label
            results.append(features)

df = pd.DataFrame(results)
print(f'\nวิเคราะห์ทั้งหมด {len(df)} รูป')
df.head(10)

## Step 5 — เปรียบเทียบค่าเฉลี่ยระหว่างดิบและสุก

In [ ]:
print('=== ค่าเฉลี่ยแต่ละ Feature ===')
print(df.groupby('class')[['black_ratio','green_ratio',
                            'yellow_ratio','spine_count',
                            'texture']].mean().round(4))

# Plot เปรียบเทียบ
features = ['black_ratio', 'green_ratio', 'yellow_ratio',
            'spine_count', 'texture']
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i, feat in enumerate(features):
    for class_name in CLASSES:
        data = df[df['class'] == class_name][feat]
        axes[i].hist(data, alpha=0.6, label=class_name, bins=20)
    axes[i].set_title(feat)
    axes[i].legend()

plt.tight_layout()
plt.show()

## Step 6 — ลอง Train classifier จาก features เหล่านี้

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = df[['black_ratio', 'green_ratio', 'yellow_ratio',
        'spine_count', 'texture']].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print('=== ผลการจำแนก ===')
print(classification_report(y_test, y_pred,
      target_names=CLASSES))

# Feature importance
feat_names = ['black_ratio', 'green_ratio', 'yellow_ratio',
              'spine_count', 'texture']
importances = clf.feature_importances_
print('\n=== Feature ที่สำคัญที่สุด ===')
for name, imp in sorted(zip(feat_names, importances),
                        key=lambda x: -x[1]):
    print(f'  {name}: {imp*100:.1f}%')